# VLM Pipeline Preprocessing Fix Test

This notebook tests the BioCLIP-2 preprocessing fix for the VLM pipeline.

**What we're testing:**
- Fixed unpacking of `preprocess_train` and `preprocess_val` from BioCLIP-2
- Switched from `preprocess_train` (with augmentation) to `preprocess_val` (for inference)
- Expected fix: predictions should have >50% confidence, not ~25% per class

## Cell 1: Setup and Check GPU

In [ ]:
# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("⚠️ WARNING: No GPU detected. Running on CPU will be slow and may have accuracy issues.")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\nUsing device: {device}")

## Cell 2: Install Dependencies

In [ ]:
# Install required packages
!pip install -q git+https://github.com/mlfoundations/open_clip.git
!pip install -q transformers pillow urllib3
!pip install -q ultralytics  # For SAM2

print("✅ Dependencies installed")

## Cell 3: Clone or Import Repo

In [ ]:
# Option A: Clone from GitHub (if available)
import os
import sys

# Try to clone repo
repo_path = '/content/aads-ulora'
if not os.path.exists(repo_path):
    print("Cloning repository...")
    os.system('git clone https://github.com/your-username/aads-ulora.git /content/aads-ulora')
    print(f"✅ Repository cloned to {repo_path}")
else:
    print(f"✅ Repository already exists at {repo_path}")

sys.path.insert(0, repo_path)

# Or if using Google Drive
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
    # Adjust path if repo is on Google Drive
    # sys.path.insert(0, '/content/gdrive/MyDrive/path-to-repo')
except:
    pass

## Cell 4: Direct BioCLIP-2 Test (The Critical Diagnostic)

In [ ]:
import logging
from PIL import Image
import urllib.request
import open_clip

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s - %(message)s'
)

print("\n" + "="*70)
print("DIRECT BioCLIP-2 TEST")
print("="*70)

# Load BioCLIP-2
print("\nLoading BioCLIP-2...")
model_id = "imageomics/bioclip-2"
hub_model_id = f"hf-hub:{model_id}"

print(f"Model ID: {hub_model_id}")
model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(hub_model_id)
tokenizer = open_clip.get_tokenizer(hub_model_id)

model = model.to(device)
model.eval()
print(f"✅ BioCLIP-2 loaded on {device}")

# Test text encoding
print("\n" + "-"*70)
print("TEXT ENCODING")
print("-"*70)

prompts = ["a photo of grape", "a photo of potato", "a photo of tomato", "a photo of strawberry"]
print(f"Prompts: {prompts}")

tokens = tokenizer(prompts).to(device)

with torch.no_grad():
    text_embeds = model.encode_text(tokens)
    print(f"\nText embeddings shape: {text_embeds.shape}")
    print(f"Text embeddings norm (before): {text_embeds.norm(dim=-1)}")
    text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)
    print(f"Text embeddings norm (after): {text_embeds.norm(dim=-1)}")

print("✅ Text encoding works correctly")

## Cell 5: Download Test Image

In [ ]:
print("\n" + "-"*70)
print("IMAGE DOWNLOAD")
print("-"*70)

# Try to download a strawberry image
strawberry_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/29/Fragaria_x_ananassa_Foto_by_CF_Weise.jpg/800px-Fragaria_x_ananassa_Foto_by_CF_Weise.jpg"
image_path = "/tmp/test_strawberry.jpg"

try:
    print(f"Downloading strawberry image...")
    urllib.request.urlretrieve(strawberry_url, image_path)
    image = Image.open(image_path).convert('RGB')
    print(f"✅ Image downloaded: {image.size}")
except Exception as e:
    print(f"Download failed: {e}")
    print(f"Using test image instead...")
    image = Image.new('RGB', (224, 224), color=(200, 0, 0))  # Red test image
    print(f"✅ Using red test image: {image.size}")

## Cell 6: Test Both Preprocesses (The Critical Comparison)

In [ ]:
print("\n" + "="*70)
print("IMAGE ENCODING TEST - CRITICAL DIAGNOSTIC")
print("="*70)
print("\nTesting BOTH preprocess_train and preprocess_val to identify the bug.")
print("If predictions differ: we found the issue!\n")

results = {}

for preprocess_name, preprocess_fn in [("TRAIN (augmentation)", preprocess_train), ("VAL (no augmentation)", preprocess_val)]:
    print("\n" + "-"*70)
    print(f"Using preprocess_{preprocess_name}")
    print("-"*70)
    
    # Preprocess image
    image_tensor = preprocess_fn(image).unsqueeze(0).to(device)
    
    print(f"\nImage tensor after preprocessing:")
    print(f"  Shape: {image_tensor.shape}")
    print(f"  DType: {image_tensor.dtype}")
    print(f"  Min: {image_tensor.min():.4f}")
    print(f"  Max: {image_tensor.max():.4f}")
    print(f"  Mean: {image_tensor.mean():.4f}")
    print(f"  Std: {image_tensor.std():.4f}")
    
    # Encode image
    with torch.no_grad():
        image_embeds = model.encode_image(image_tensor)
        print(f"\nImage embeddings (before normalization):")
        print(f"  Shape: {image_embeds.shape}")
        print(f"  Norm: {image_embeds.norm(dim=-1).item():.4f}")
        print(f"  Mean: {image_embeds.mean():.4f}")
        
        # Normalize
        image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
        print(f"\nImage embeddings (after normalization):")
        print(f"  Norm: {image_embeds.norm(dim=-1).item():.4f}")
        
        # Compute similarity
        logits = image_embeds @ text_embeds.T
        probs = torch.softmax(logits, dim=-1)
        
        print(f"\nSimilarity scores (logits): {logits.squeeze()}")
        print(f"Probabilities: {probs.squeeze()}")
        
        # Get prediction
        pred_idx = torch.argmax(probs)
        pred_conf = probs[0, pred_idx].item()
        pred_label = prompts[pred_idx].replace("a photo of ", "")
        
        print(f"\n🎯 Prediction: {pred_label} ({pred_conf:.1%})")
        results[preprocess_name] = (pred_label, pred_conf, probs.squeeze().cpu())

## Cell 7: Analysis - Did We Fix It?

In [ ]:
print("\n" + "="*70)
print("ANALYSIS - DID THE FIX WORK?")
print("="*70)

train_label, train_conf, train_probs = results["TRAIN (augmentation)"]
val_label, val_conf, val_probs = results["VAL (no augmentation)"]

print(f"\npreprocess_train result: {train_label} ({train_conf:.1%})")
print(f"  Probabilities: {train_probs}")
print(f"  Max confidence: {train_probs.max():.1%}")

print(f"\npreprocess_val result: {val_label} ({val_conf:.1%})")
print(f"  Probabilities: {val_probs}")
print(f"  Max confidence: {val_probs.max():.1%}")

# Check for the bug pattern
train_is_random = (train_probs - 0.25).abs().max() < 0.02  # All ~25%
val_is_random = (val_probs - 0.25).abs().max() < 0.02  # All ~25%

print("\n" + "-"*70)
if train_is_random and not val_is_random:
    print("✅ FIX CONFIRMED!")
    print(f"  preprocess_train shows random 25% (BUG) - as expected")
    print(f"  preprocess_val shows {val_probs.max():.1%} (FIXED) - preprocessing works!")
elif val_is_random:
    print("❌ STILL BROKEN")
    print(f"  Even preprocess_val shows ~25% per class (random)")
    print(f"  This suggests a different problem than preprocessing")
elif train_conf > val_conf:
    print("⚠️ UNEXPECTED RESULT")
    print(f"  preprocess_train ({train_conf:.1%}) > preprocess_val ({val_conf:.1%})")
    print(f"  This is backwards from what we expect")
else:
    print("✅ preprocess_val WORKING BETTER than preprocess_train")
    print(f"  Improvement: {val_conf - train_conf:.1%}")

## Cell 8: Test Full VLM Pipeline

In [ ]:
print("\n" + "="*70)
print("FULL VLM PIPELINE TEST")
print("="*70)

try:
    from src.router.vlm_pipeline import VLMPipeline
    
    # Configuration
    config = {
        'vlm_enabled': True,
        'vlm_confidence_threshold': 0.5,
        'vlm_max_detections': 10,
        'vlm_strict_model_loading': False,
        'vlm_model_source': 'huggingface'
    }
    
    # Initialize and load
    print("\nInitializing VLM pipeline...")
    pipeline = VLMPipeline(config, device=device)
    
    print("Loading models...")
    pipeline.load_models()
    
    if pipeline.is_ready():
        print("✅ Pipeline ready")
        
        # Test crop classification
        print(f"\nCrop labels: {pipeline.crop_labels}")
        crop_label, crop_conf = pipeline._classify_with_preencoded(image, 'crop')
        print(f"\n🎯 Pipeline Prediction: {crop_label} ({crop_conf:.1%})")
        
        # Compare with direct test
        print(f"\nComparison:")
        print(f"  Direct BioCLIP-2 (preprocess_val): {val_label} ({val_conf:.1%})")
        print(f"  VLM Pipeline: {crop_label} ({crop_conf:.1%})")
        
        if crop_label == val_label:
            print(f"  ✅ MATCH! Pipeline using correct preprocessing")
        else:
            print(f"  ⚠️ MISMATCH! Different predictions")
    else:
        print("❌ Pipeline not ready")
        
except Exception as e:
    print(f"❌ Error loading VLM pipeline: {e}")
    import traceback
    traceback.print_exc()

## Cell 9: Summary and Next Steps

In [ ]:
print("\n" + "="*70)
print("TEST SUMMARY")
print("="*70)

print(f"\nGPU Used: {device}")
print(f"Test Image: Strawberry (or substitute)")

print(f"\nResults:")
print(f"  Text Encoding: ✅ Works correctly")
print(f"  preprocess_train: {train_label} ({train_conf:.1%})")
print(f"  preprocess_val: {val_label} ({val_conf:.1%})")

if not train_is_random and val_conf > 0.4:
    print(f"\n✅ SUCCESS! The preprocessing fix appears to be working.")
elif train_is_random and val_conf > 0.4:
    print(f"\n✅ EXCELLENT! Fixed the preprocessing bug!")
    print(f"  preprocess_train was showing random ~25%")
    print(f"  preprocess_val now shows correct {val_conf:.1%}")
else:
    print(f"\n⚠️ Something still isn't right. Check the debug output above.")

print(f"\nNext steps:")
print(f"1. Copy this test output and share it")
print(f"2. Verify GPU is being used (device should be 'cuda')")
print(f"3. Check if preprocess_val shows >50% confidence")
print(f"4. If still broken, check image tensor min/max values - should be in [-1, 1] or [0, 1]")